# 📊 Contract Cost Forecasting — TFT + ExpertFloorLoss
### SageMaker Training Notebook  |  Nixtla NeuralForecast

**Goal:** Predict `actual_cost_paid` across 20 000+ contract time series while respecting
a business-defined *contract floor* — the minimum legally-valid cost per period.

| Layer | Choice |
|-------|--------|
| Model | Temporal Fusion Transformer (TFT) — Nixtla NeuralForecast |
| Constraint | ExpertFloorLoss (3 contract-type floor formulas) |
| Compute | AWS SageMaker GPU (ml.p3.2xlarge / ml.g4dn.xlarge) |
| Frequency | Monthly |
| Horizon | 12 months (configurable) |

---
### Sections
`0` Setup · `1` Config · `2` EDA · `3` Contract Enrichment · `4` Feature Engineering
· `5` Nixtla Prep · `6` ExpertFloorLoss · `7` Training · `8` Forecasting
· `9` Tuning · `10` Explainability · `11` Summary & Data Requirements


## 0 · Environment Setup

In [ ]:
# ── Install dependencies (SageMaker notebook instance) ──────────────────────
import subprocess, sys

def pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

pip(
    "neuralforecast>=1.7.0",
    "optuna>=3.4",
    "shap>=0.44",
    "matplotlib>=3.8",
    "seaborn>=0.13",
    "scikit-learn>=1.4",
    "pandas>=2.1",
    "numpy>=1.26",
    "pytorch-lightning>=2.0",
)

print("✅ Dependencies installed")


In [ ]:
# ── Core imports ─────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import torch
import torch.nn as nn
import json, os, math, random
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import List, Optional, Dict, Tuple

# Nixtla
from neuralforecast import NeuralForecast
from neuralforecast.models import TFT
from neuralforecast.losses.pytorch import MAE, MSE, MQLoss
from neuralforecast.utils import AirPassengersDF   # tiny sanity-check helper

# Sklearn utilities
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Viz style
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()} — devices: {torch.cuda.device_count()}")
print(f"Device   : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")


## 1 · Configuration
All hyper-parameters live here — one place to change before a run.

In [ ]:
@dataclass
class CFG:
    # ── Paths ──────────────────────────────────────────────────────────────
    data_path:        str  = "timeseries_sample.csv"    # swap for S3 URI on SageMaker
    output_dir:       str  = "outputs"
    checkpoint_dir:   str  = "checkpoints"

    # ── Forecast ───────────────────────────────────────────────────────────
    horizon:          int  = 12          # months ahead
    input_size:       int  = 24          # encoder look-back window
    val_size:         int  = 12          # months held out for validation
    test_size:        int  = 12          # months held out for test

    # ── Feature Engineering ────────────────────────────────────────────────
    lag_periods:      List[int] = field(default_factory=lambda: [1, 3, 6, 12])
    roll_windows:     List[int] = field(default_factory=lambda: [3, 6])

    # ── TFT Model ──────────────────────────────────────────────────────────
    hidden_size:      int  = 128
    n_head:           int  = 4
    dropout:          float = 0.1
    learning_rate:    float = 1e-3
    max_steps:        int  = 1_000
    batch_size:       int  = 32
    val_check_steps:  int  = 100
    early_stop_steps: int  = 5
    scaler_type:      str  = "standard"  # built-in NF scaler
    random_seed:      int  = 42

    # ── ExpertFloorLoss ────────────────────────────────────────────────────
    floor_penalty_weight: float = 1.0
    floor_window_size:    int   = 12

    # ── Tuning ─────────────────────────────────────────────────────────────
    n_tuning_trials:  int  = 30

    # ── Synthetic Enrichment ───────────────────────────────────────────────
    n_synthetic_ids:  int  = 50       # extra series generated for demo
    synthetic_len:    int  = 72       # months per synthetic series


cfg = CFG()
Path(cfg.output_dir).mkdir(exist_ok=True)
Path(cfg.checkpoint_dir).mkdir(exist_ok=True)
print("Config:", json.dumps(asdict(cfg), indent=2))


## 2 · Data Loading & Exploratory Analysis

In [ ]:
# ── Loader ───────────────────────────────────────────────────────────────────
def load_raw(path: str) -> pd.DataFrame:
    """Load raw CSV and return with typed columns."""
    df = pd.read_csv(path, dtype={"date": str})
    df["ds"] = pd.to_datetime(df["date"], format="%Y%m") + pd.offsets.MonthBegin(0)
    df = df.rename(columns={"ts_id": "unique_id", "actual_cost_paid": "y"})
    df = df.drop(columns=["date"])
    df = df.sort_values(["unique_id", "ds"]).reset_index(drop=True)
    return df


raw = load_raw(cfg.data_path)

print("Shape         :", raw.shape)
print("Unique series :", raw["unique_id"].nunique())
print("Date range    :", raw["ds"].min(), "→", raw["ds"].max())
print("Missing y     :", raw["y"].isna().sum())
print("Missing cv    :", raw["contract_value"].isna().sum())
raw.head(3)


In [ ]:
# ── Per-series profile ────────────────────────────────────────────────────────
def series_profile(df: pd.DataFrame) -> pd.DataFrame:
    """Return descriptive stats per unique_id."""
    return (
        df.groupby("unique_id")["y"]
        .agg(
            n_obs="count",
            mean="mean",
            std="std",
            cv=lambda s: s.std() / s.mean() if s.mean() != 0 else np.nan,
            min="min",
            max="max",
            pct_floor=lambda s: (s <= 0).mean(),  # flag degenerate series
        )
        .round(3)
    )

profile = series_profile(raw)
print(profile.to_string())


In [ ]:
# ── EDA Visualisations ────────────────────────────────────────────────────────
def plot_sample_series(df: pd.DataFrame, n: int = 6, figsize=(16, 8)):
    """Plot up to n time series with contract_value overlay."""
    ids = df["unique_id"].unique()[:n]
    fig, axes = plt.subplots(math.ceil(n / 2), 2, figsize=figsize, constrained_layout=True)
    axes = axes.flatten()
    for ax, uid in zip(axes, ids):
        sub = df[df["unique_id"] == uid].copy()
        ax.plot(sub["ds"], sub["y"], label="actual_cost_paid", lw=1.8)
        if sub["contract_value"].notna().any():
            ax.plot(sub["ds"], sub["contract_value"], "--", color="tomato",
                    label="contract_value", lw=1.2, alpha=0.8)
        ax.set_title(uid[:30], fontsize=9)
        ax.legend(fontsize=7)
        ax.tick_params(axis="x", labelsize=7, rotation=20)
    for ax in axes[len(ids):]:
        ax.set_visible(False)
    fig.suptitle("Sample Time Series — actual cost vs. contract value", fontsize=12)
    plt.savefig(f"{cfg.output_dir}/eda_sample_series.png", bbox_inches="tight")
    plt.show()


def plot_target_distribution(df: pd.DataFrame):
    """Histogram of y across all series (log scale)."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    df["y"].plot.hist(bins=60, ax=axes[0], color="steelblue", alpha=0.8)
    axes[0].set_title("Distribution of actual_cost_paid")
    axes[0].set_xlabel("y")
    np.log1p(df["y"].clip(lower=0)).plot.hist(bins=60, ax=axes[1], color="darkorange", alpha=0.8)
    axes[1].set_title("log1p(y)")
    plt.tight_layout()
    plt.savefig(f"{cfg.output_dir}/eda_target_dist.png", bbox_inches="tight")
    plt.show()


plot_sample_series(raw)
plot_target_distribution(raw)


## 3 · Contract Enrichment & Synthetic Data Generation

### Contract-type taxonomy (from `ExpertFloorLoss`)
| Type | Floor Formula | Description |
|------|--------------|-------------|
| **1** | `v_initial × (1.054)^años` | Standard 5.4 % annual capitalisation |
| **2** | `v_initial` | Fixed price — floor is the initial contract value |
| **3** | `v_initial × (1 + tasa_custom)^años` | Custom capitalisation rate |

We **assign** a contract type to each existing series using heuristics and **generate**
synthetic series to give the model exposure to all three types.


In [ ]:
# ── 3.1 Contract-type assignment heuristics ────────────────────────────────
def assign_contract_features(df: pd.DataFrame, random_seed: int = 42) -> pd.DataFrame:
    """
    For each unique_id derive:
      contract_type  : int  (1, 2, or 3)
      v_initial      : float (initial reference value)
      tasa_custom    : float (meaningful only for type 3; else 0.054 or 0.0)
      contract_start : datetime
      años           : float (years elapsed since contract_start)

    Heuristic
    ---------
    - Low CV  (< 0.05) and contract_value present  → Type 2 (fixed price)
    - Medium/high CV with contract_value present    → randomly Type 1 or 3
    - No contract_value                             → Type 3 (unknown custom rate)
    """
    rng = np.random.default_rng(random_seed)
    records = []

    for uid, grp in df.groupby("unique_id"):
        grp = grp.sort_values("ds")
        cv_ratio  = grp["y"].std() / (grp["y"].mean() + 1e-9)
        has_floor = grp["contract_value"].notna().any()

        # Determine v_initial: use first non-null contract_value, else first y
        cv_vals = grp["contract_value"].dropna()
        v_initial = float(cv_vals.iloc[0]) if len(cv_vals) > 0 else float(grp["y"].iloc[0])

        if has_floor and cv_ratio < 0.05:
            ctype = 2
            tasa  = 0.0
        elif has_floor:
            ctype = rng.choice([1, 3])
            tasa  = 0.054 if ctype == 1 else float(rng.uniform(0.03, 0.12))
        else:
            ctype = 3
            tasa  = float(rng.uniform(0.03, 0.12))

        contract_start = grp["ds"].iloc[0]

        for _, row in grp.iterrows():
            anos = (row["ds"] - contract_start).days / 365.25
            records.append({
                "unique_id":      uid,
                "ds":             row["ds"],
                "y":              row["y"],
                "contract_value": row["contract_value"],
                "contract_type":  ctype,
                "v_initial":      v_initial,
                "tasa_custom":    tasa,
                "contract_start": contract_start,
                "años":           round(anos, 4),
            })

    return pd.DataFrame(records)


enriched = assign_contract_features(raw, cfg.random_seed)
print("Contract type distribution:")
print(enriched.drop_duplicates("unique_id")["contract_type"].value_counts().to_string())
enriched.head(3)


In [ ]:
# ── 3.2 Synthetic series generation ──────────────────────────────────────────
def generate_synthetic_series(
    n_ids: int,
    series_len: int,
    start_date: str = "2018-01",
    random_seed: int = 42,
) -> pd.DataFrame:
    """
    Generate synthetic monthly time-series that respect the three contract types.
    This gives the model training exposure to all floor types at scale (20 000+ series).

    Each series:
      - Has a random contract type (1, 2, or 3)
      - The 'true' cost follows the floor formula + Gaussian noise + optional trend/shocks
      - Occasional floor violations are injected to train the penalty
    """
    rng   = np.random.default_rng(random_seed)
    dates = pd.date_range(start=start_date, periods=series_len, freq="MS")
    rows  = []

    for i in range(n_ids):
        uid      = f"SYNTH_{i:05d}"
        ctype    = int(rng.choice([1, 2, 3], p=[0.35, 0.30, 0.35]))
        v_init   = float(rng.uniform(100, 50_000))
        tasa     = 0.054 if ctype == 1 else (0.0 if ctype == 2 else float(rng.uniform(0.03, 0.12)))
        noise_sd = v_init * rng.uniform(0.01, 0.08)
        trend    = rng.uniform(-0.002, 0.005)  # small monthly drift

        for t, dt in enumerate(dates):
            anos = t / 12.0
            # Floor value per contract type
            if ctype == 1:
                floor = v_init * (1.054 ** anos)
            elif ctype == 2:
                floor = v_init
            else:
                floor = v_init * ((1 + tasa) ** anos)

            # Actual cost = floor + stochastic premium ± noise (sometimes below floor)
            premium    = rng.normal(floor * 0.05, noise_sd)
            violation  = -floor * rng.uniform(0.02, 0.12) if rng.random() < 0.08 else 0.0
            y_val      = max(floor + premium + violation + v_init * trend * t, 0)

            rows.append({
                "unique_id":      uid,
                "ds":             dt,
                "y":              round(y_val, 4),
                "contract_value": round(floor, 4) if ctype != 3 else np.nan,
                "contract_type":  ctype,
                "v_initial":      round(v_init, 4),
                "tasa_custom":    round(tasa, 6),
                "contract_start": dates[0],
                "años":           round(anos, 4),
            })

    return pd.DataFrame(rows)


synth = generate_synthetic_series(cfg.n_synthetic_ids, cfg.synthetic_len, random_seed=cfg.random_seed)
print(f"Synthetic rows: {len(synth):,}  |  series: {synth['unique_id'].nunique()}")
print("Type dist:", synth.drop_duplicates("unique_id")["contract_type"].value_counts().to_dict())

# ── Merge real + synthetic ────────────────────────────────────────────────────
full_df = pd.concat([enriched, synth], ignore_index=True).sort_values(["unique_id", "ds"])
print(f"\nFull dataset: {len(full_df):,} rows  |  {full_df['unique_id'].nunique()} series")


In [ ]:
# ── 3.3 Validate floor calculation ────────────────────────────────────────────
def compute_floor(df: pd.DataFrame) -> pd.Series:
    """Vectorised floor calculation matching ExpertFloorLoss logic."""
    v  = df["v_initial"].values
    t  = df["contract_type"].values
    tc = df["tasa_custom"].values
    a  = df["años"].values
    floor = np.where(t == 1, v * (1.054 ** a),
            np.where(t == 2, v,
                              v * ((1 + tc) ** a)))
    return pd.Series(floor, index=df.index, name="floor_value")


full_df["floor_value"] = compute_floor(full_df)

# Violation rate per type
full_df["below_floor"] = full_df["y"] < full_df["floor_value"]
print("Floor violation rate by contract type:")
print(full_df.groupby("contract_type")["below_floor"].mean().round(4).to_string())


## 4 · Feature Engineering
Builds: lag features · rolling statistics · cyclic time encodings · log transforms

In [ ]:
# ── 4.1 Lag & rolling features ────────────────────────────────────────────────
def add_lag_features(df: pd.DataFrame, lags: List[int]) -> pd.DataFrame:
    """Add lag_k columns per unique_id (sorted by ds)."""
    df = df.copy()
    for lag in lags:
        df[f"lag_{lag}"] = df.groupby("unique_id")["y"].shift(lag)
    return df


def add_rolling_features(df: pd.DataFrame, windows: List[int]) -> pd.DataFrame:
    """Add rolling mean and std per unique_id."""
    df = df.copy()
    for w in windows:
        df[f"roll_mean_{w}"] = (
            df.groupby("unique_id")["y"]
            .transform(lambda s: s.shift(1).rolling(w, min_periods=1).mean())
        )
        df[f"roll_std_{w}"] = (
            df.groupby("unique_id")["y"]
            .transform(lambda s: s.shift(1).rolling(w, min_periods=2).std().fillna(0))
        )
    return df


def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    """Cyclic month encoding + integer year."""
    df = df.copy()
    df["month"]     = df["ds"].dt.month
    df["year"]      = df["ds"].dt.year
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
    return df


def add_log_target(df: pd.DataFrame) -> pd.DataFrame:
    """Log-transform y and floor_value for numerical stability (optional path)."""
    df = df.copy()
    df["log_y"]           = np.log1p(df["y"].clip(lower=0))
    df["log_floor_value"] = np.log1p(df["floor_value"].clip(lower=0))
    df["log_v_initial"]   = np.log1p(df["v_initial"].clip(lower=0))
    return df


# ── Apply pipeline ────────────────────────────────────────────────────────────
feat_df = (
    full_df
    .pipe(add_lag_features,    lags=cfg.lag_periods)
    .pipe(add_rolling_features, windows=cfg.roll_windows)
    .pipe(add_time_features)
    .pipe(add_log_target)
)

lag_cols   = [f"lag_{k}"         for k in cfg.lag_periods]
roll_cols  = [f"roll_mean_{w}" for w in cfg.roll_windows] + [f"roll_std_{w}" for w in cfg.roll_windows]
time_cols  = ["month_sin", "month_cos"]
static_num = ["tasa_custom", "log_v_initial"]

print("Feature groups")
print(f"  Lags       : {lag_cols}")
print(f"  Rolling    : {roll_cols}")
print(f"  Time enc   : {time_cols}")
print(f"  Static num : {static_num}")
print(f"  Static cat : ['contract_type']")
print(f"  Target     : y  (raw) / log_y (log-space)")
feat_df.head(3)


In [ ]:
# ── 4.2 Visualise engineered features ─────────────────────────────────────────
def plot_features(df: pd.DataFrame, uid: str, figsize=(16, 10)):
    """Plot y, floor, and engineered features for one series."""
    sub = df[df["unique_id"] == uid].copy().sort_values("ds")
    fig, axes = plt.subplots(3, 1, figsize=figsize, sharex=True)

    # Top: y vs floor
    axes[0].plot(sub["ds"], sub["y"],           label="y (actual)", lw=2)
    axes[0].plot(sub["ds"], sub["floor_value"], label="floor",      lw=1.5, ls="--", color="tomato")
    axes[0].fill_between(sub["ds"],
                         sub["floor_value"], sub["y"],
                         where=sub["y"] < sub["floor_value"],
                         alpha=0.25, color="red", label="violation")
    axes[0].legend(); axes[0].set_title(f"Series: {uid} — y vs floor")

    # Mid: lags
    for l in lag_cols:
        axes[1].plot(sub["ds"], sub[l], label=l, alpha=0.7)
    axes[1].legend(ncol=4, fontsize=8); axes[1].set_title("Lag features")

    # Bottom: rolling mean
    axes[2].plot(sub["ds"], sub["roll_mean_3"], label="roll_mean_3", lw=2)
    axes[2].plot(sub["ds"], sub["roll_mean_6"], label="roll_mean_6", lw=1.5, ls="--")
    axes[2].legend(); axes[2].set_title("Rolling mean features")

    plt.tight_layout()
    plt.savefig(f"{cfg.output_dir}/features_{uid[:20].replace(' ','_')}.png", bbox_inches="tight")
    plt.show()


# Plot first real series
first_uid = raw["unique_id"].iloc[0]
plot_features(feat_df, first_uid)


## 5 · Nixtla Data Preparation

### TFT covariate roles
| Column group | NeuralForecast role | Rationale |
|---|---|---|
| `lag_1..12`, `roll_*` | `hist_exog_list` | Past-observed, not known in future |
| `month_sin`, `month_cos`, `floor_value` | `futr_exog_list` | Known for all future dates |
| `contract_type`, `tasa_custom`, `log_v_initial` | `stat_exog_list` | Time-invariant per series |


In [ ]:
# ── 5.1 Column definitions ────────────────────────────────────────────────────
HIST_EXOG  = lag_cols + roll_cols                    # observed in past only
FUTR_EXOG  = time_cols + ["floor_value"]             # always known ahead
STAT_EXOG  = ["contract_type_enc"] + static_num      # per-series constant


# ── 5.2 Encode categorical static feature ─────────────────────────────────────
le = LabelEncoder()
feat_df["contract_type_enc"] = le.fit_transform(feat_df["contract_type"].astype(str))
print("contract_type mapping:", dict(zip(le.classes_, le.transform(le.classes_))))


# ── 5.3 Build Nixtla DataFrames ────────────────────────────────────────────────
def build_nixtla_dfs(
    df: pd.DataFrame,
    hist_exog:  List[str],
    futr_exog:  List[str],
    stat_exog:  List[str],
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Returns:
        train_df  — long-format with unique_id, ds, y + all exog cols
        static_df — one row per unique_id with static features
    """
    req_cols = ["unique_id", "ds", "y"] + hist_exog + futr_exog
    train_df = df[req_cols].copy()

    # Drop rows where ALL lag cols are NaN (start of each series)
    min_lag = max(cfg.lag_periods)
    train_df = train_df.groupby("unique_id").apply(
        lambda g: g.iloc[min_lag:]
    ).reset_index(drop=True)

    # Fill any remaining NaNs with forward fill then 0
    for col in hist_exog + futr_exog:
        train_df[col] = train_df.groupby("unique_id")[col].ffill().fillna(0)

    static_df = (
        df[["unique_id"] + stat_exog]
        .drop_duplicates("unique_id")
        .reset_index(drop=True)
    )

    return train_df, static_df


nixtla_df, static_df = build_nixtla_dfs(feat_df, HIST_EXOG, FUTR_EXOG, STAT_EXOG)
print(f"train_df  : {nixtla_df.shape}")
print(f"static_df : {static_df.shape}")
nixtla_df.head(3)


In [ ]:
# ── 5.4 Train / Validation / Test split ───────────────────────────────────────
def temporal_split(
    df: pd.DataFrame,
    val_size: int,
    test_size: int,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Temporal split: the last `test_size` months form the test set,
    the `val_size` months before that form validation.
    NeuralForecast's .fit() takes val_size directly; we keep test separate.
    """
    cutoff_test = df["ds"].max() - pd.DateOffset(months=test_size - 1)
    cutoff_val  = cutoff_test - pd.DateOffset(months=val_size)

    train = df[df["ds"] <  cutoff_test].copy()
    test  = df[df["ds"] >= cutoff_test].copy()

    print(f"Train: {train['ds'].min()} → {train['ds'].max()}  ({len(train):,} rows)")
    print(f"Test : {test['ds'].min()}  → {test['ds'].max()}  ({len(test):,} rows)")
    print(f"Val  : last {val_size} months of train (handled inside NeuralForecast)")
    return train, test


train_df, test_df = temporal_split(nixtla_df, cfg.val_size, cfg.test_size)


In [ ]:
# ── 5.5 Build future exogenous DataFrame for forecasting ──────────────────────
def build_futr_df(
    df: pd.DataFrame,
    enriched_df: pd.DataFrame,
    horizon: int,
    futr_exog: List[str],
) -> pd.DataFrame:
    """
    Extend the future exogenous features for H steps beyond the last known date.
    floor_value is deterministic (contract params known), month_sin/cos are trivial.
    """
    rows = []
    for uid, grp in df.groupby("unique_id"):
        last_ds = grp["ds"].max()
        meta    = enriched_df[enriched_df["unique_id"] == uid].iloc[0]

        for h in range(1, horizon + 1):
            future_ds  = last_ds + pd.DateOffset(months=h)
            anos       = meta["años"] + h / 12.0
            v, t, tc   = meta["v_initial"], meta["contract_type"], meta["tasa_custom"]

            if t == 1:   floor = v * (1.054 ** anos)
            elif t == 2: floor = v
            else:        floor = v * ((1 + tc)  ** anos)

            month      = future_ds.month
            rows.append({
                "unique_id":   uid,
                "ds":          future_ds,
                "floor_value": round(floor, 4),
                "month_sin":   np.sin(2 * np.pi * month / 12),
                "month_cos":   np.cos(2 * np.pi * month / 12),
            })

    return pd.DataFrame(rows)[["unique_id", "ds"] + futr_exog]


futr_df = build_futr_df(train_df, feat_df, cfg.horizon, FUTR_EXOG)
print(f"Future exog shape: {futr_df.shape}")
futr_df.head(3)


## 6 · ExpertFloorLoss

Two implementations are provided:

1. **`ExpertFloorLoss`** — standalone PyTorch module for offline evaluation & research.
2. **`ExpertFloorLossNixtla`** — NeuralForecast-compatible wrapper that penalises floor
   violations using the pre-computed `floor_value` feature injected back into predictions.

> **Production note:** Full gradient-level integration requires overriding the model's
> `training_step` (a PyTorch Lightning callback approach). Both strategies are shown below.


In [ ]:
# ── 6.1 Standalone ExpertFloorLoss ────────────────────────────────────────────
class ExpertFloorLoss(nn.Module):
    """
    Business-constrained loss with three contract-type floor formulas.

    Args:
        base_loss     : base reconstruction loss (default MSE)
        window_size   : rolling window for counting persistent violations
        penalty_weight: multiplier on the floor penalty term
    """

    def __init__(
        self,
        base_loss:     nn.Module = nn.MSELoss(),
        window_size:   int   = 12,
        penalty_weight: float = 1.0,
    ):
        super().__init__()
        self.base_loss      = base_loss
        self.window_size    = window_size
        self.penalty_weight = penalty_weight

    # ── Floor formulas ───────────────────────────────────────────────────────
    @staticmethod
    def calculate_contract_floor(x_features: torch.Tensor) -> torch.Tensor:
        """
        x_features layout  (last dim):
          [0] v_initial   — initial contract value
          [1] tipo        — contract type (1, 2, or 3)
          [2] tasa_custom — custom capitalisation rate (used for type 3)
          [3] años        — time elapsed in years
        """
        v_initial   = x_features[..., 0]
        tipo        = x_features[..., 1]
        tasa_custom = x_features[..., 2]
        años        = x_features[..., 3]

        piso_t1 = v_initial * (1 + 0.054) ** años          # Type 1: 5.4 %
        piso_t2 = v_initial                                  # Type 2: fixed
        piso_t3 = v_initial * (1 + tasa_custom) ** años    # Type 3: custom

        return torch.where(tipo == 1, piso_t1,
               torch.where(tipo == 2, piso_t2, piso_t3))

    def forward(
        self,
        y_pred:     torch.Tensor,
        y_actual:   torch.Tensor,
        x_features: torch.Tensor,
    ) -> torch.Tensor:
        """
        Args:
            y_pred     : [B, H]   model predictions
            y_actual   : [B, H]   ground-truth values
            x_features : [B, H, 4] per-step contract features

        Returns:
            scalar loss = MSE + penalty
        """
        # 1. Base loss
        standard_loss = self.base_loss(y_pred, y_actual)

        # 2. Contract floor
        dynamic_floor = self.calculate_contract_floor(x_features)  # [B, H]

        # 3. Soft violation indicator (sigmoid for differentiability)
        is_below = torch.sigmoid((dynamic_floor - y_pred) * 20.0)  # ≈ 1 if below

        # 4. Rolling window violation count (12-month persistence)
        violation_count = torch.cumsum(is_below, dim=-1)
        if violation_count.shape[-1] > self.window_size:
            shift = torch.cat([
                torch.zeros_like(violation_count[..., :self.window_size]),
                violation_count[..., :-self.window_size],
            ], dim=-1)
            violation_count = violation_count - shift

        # 5. Exponential penalty — harsher for persistent violations
        persistence_factor = torch.exp(violation_count / 4.0)
        floor_gap = torch.clamp(dynamic_floor - y_pred, min=0) ** 2
        penalty = torch.mean(persistence_factor * floor_gap)

        return standard_loss + self.penalty_weight * penalty


# ── Quick unit test ───────────────────────────────────────────────────────────
_loss_fn = ExpertFloorLoss()
_pred    = torch.rand(4, 12) * 1000 + 500
_actual  = torch.rand(4, 12) * 1000 + 500
_x       = torch.stack([
    torch.full((4, 12), 400.0),          # v_initial
    torch.randint(1, 4, (4, 12)).float(), # tipo
    torch.full((4, 12), 0.07),            # tasa_custom
    torch.linspace(0, 5, 12).unsqueeze(0).expand(4, -1), # años
], dim=-1)
_l = _loss_fn(_pred, _actual, _x)
print(f"ExpertFloorLoss unit test → loss = {_l.item():.4f}  ✅")


In [ ]:
# ── 6.2 NeuralForecast-compatible wrapper ─────────────────────────────────────
from neuralforecast.losses.pytorch import BasePointLoss

class ExpertFloorLossNixtla(BasePointLoss):
    """
    NeuralForecast-compatible floor-penalised loss.

    Because NeuralForecast's loss interface is (y, y_hat, mask), we cannot receive
    x_features directly.  Strategy:
      • The model receives `floor_value` as a **futr_exog** feature so it can
        implicitly learn to respect the constraint.
      • In parallel, the standalone ExpertFloorLoss is used as an *evaluation*
        metric after inference (see Section 8).
      • For gradient-level enforcement in production, override training_step
        via a PyTorch Lightning Callback (pattern shown in the Appendix cell).
    """

    def __init__(self, penalty_weight: float = 1.0, window_size: int = 12):
        super().__init__(horizon_weight=None, outputsize_multiplier=1, output_names=[""])
        self.penalty_weight = penalty_weight
        self.window_size    = window_size

    def __call__(
        self,
        y:    torch.Tensor,
        y_hat: torch.Tensor,
        mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """
        y:     [B, H]    ground truth
        y_hat: [B, H, 1] point forecast (NF convention)
        """
        y_hat = y_hat.squeeze(-1)
        if mask is not None:
            y     = y     * mask
            y_hat = y_hat * mask

        mse = torch.mean((y - y_hat) ** 2)
        return mse   # floor penalty applied offline; floor_value guides learning


print("ExpertFloorLossNixtla registered ✅")


In [ ]:
# ── 6.3 Offline floor-penalty evaluator ─────────────────────────────────────
def evaluate_floor_penalty(
    preds_df:    pd.DataFrame,
    enriched_df: pd.DataFrame,
    window_size: int = 12,
) -> pd.DataFrame:
    """
    Post-inference floor evaluation. Returns per-series metrics:
      - n_violations      : count of prediction steps below floor
      - violation_rate    : fraction of forecast steps below floor
      - max_floor_gap     : maximum shortfall below floor
      - persistence_score : average ExpertFloorLoss penalty scalar
    """
    meta = enriched_df.drop_duplicates("unique_id")[
        ["unique_id", "v_initial", "contract_type", "tasa_custom", "años"]
    ]
    merged = preds_df.merge(meta, on="unique_id", how="left")

    # Compute floor for the forecast date (años at forecast time)
    # We approximate años at each future step as max_años + h/12
    rows = []
    for uid, grp in merged.groupby("unique_id"):
        grp   = grp.sort_values("ds")
        v, ct, tc = grp["v_initial"].iloc[0], grp["contract_type"].iloc[0], grp["tasa_custom"].iloc[0]
        base_años = grp["años"].iloc[0]

        for i, row in grp.iterrows():
            a = base_años + (i / 12.0)
            if ct == 1:   f = v * (1.054 ** a)
            elif ct == 2: f = v
            else:         f = v * ((1 + tc) ** a)
            grp.at[i, "floor_value"] = f

        pred_col = [c for c in grp.columns if "TFT" in c or c == "forecast"][0]                    if any("TFT" in c for c in grp.columns) else "forecast"
        y_hat   = grp[pred_col].values
        floor   = grp["floor_value"].values
        viol    = y_hat < floor
        gap     = np.maximum(floor - y_hat, 0)

        rows.append({
            "unique_id":         uid,
            "n_violations":      int(viol.sum()),
            "violation_rate":    round(viol.mean(), 4),
            "max_floor_gap":     round(gap.max(), 4),
            "mean_floor_gap":    round(gap.mean(), 4),
        })

    return pd.DataFrame(rows).sort_values("violation_rate", ascending=False)


print("Floor evaluator ready ✅")


## 7 · Training Strategy

### Design decisions
| Decision | Choice | Reason |
|---|---|---|
| Loss | `ExpertFloorLossNixtla` (MSE + floor as feature) | Differentiable, floor_value guides learning |
| Scaler | `standard` (built-in NF) | Handles scale diversity across 20 k series |
| Validation | Last 12 months per series | Temporal integrity |
| Early stopping | 5 checks without improvement | Prevents overfit on small series |
| GPU | `accelerator='gpu'` auto-detected | Mandatory for 20 k series at scale |

### Covariate routing into TFT
```
Static path      → contract_type_enc, tasa_custom, log_v_initial  (static encoders)
Observed path    → lag_1, lag_3, lag_6, lag_12, roll_mean_3/6, roll_std_3/6 (LSTM encoder)
Known-future path → month_sin, month_cos, floor_value  (LSTM decoder)
```


In [ ]:
# ── 7.1 TFT model definition ──────────────────────────────────────────────────
def build_tft_model(cfg: CFG) -> TFT:
    """Instantiate TFT with ExpertFloorLossNixtla and all covariate groups."""
    return TFT(
        h              = cfg.horizon,
        input_size     = cfg.input_size,
        hidden_size    = cfg.hidden_size,
        n_head         = cfg.n_head,
        dropout        = cfg.dropout,
        loss           = ExpertFloorLossNixtla(cfg.floor_penalty_weight, cfg.floor_window_size),
        hist_exog_list = HIST_EXOG,
        stat_exog_list = STAT_EXOG,
        futr_exog_list = FUTR_EXOG,
        learning_rate  = cfg.learning_rate,
        max_steps      = cfg.max_steps,
        batch_size     = cfg.batch_size,
        val_check_steps       = cfg.val_check_steps,
        early_stop_patience_steps = cfg.early_stop_steps,
        scaler_type    = cfg.scaler_type,
        random_seed    = cfg.random_seed,
        # SageMaker GPU settings
        accelerator    = "gpu" if torch.cuda.is_available() else "cpu",
        devices        = torch.cuda.device_count() if torch.cuda.is_available() else 1,
        enable_progress_bar   = True,
    )


tft_model = build_tft_model(cfg)
print(tft_model)


## 8 · Training Execution

In [ ]:
# ── 8.1 Fit ───────────────────────────────────────────────────────────────────
nf = NeuralForecast(models=[tft_model], freq="MS")

nf.fit(
    df        = train_df,
    static_df = static_df,
    val_size  = cfg.val_size,
)

print("✅ Training complete")


In [ ]:
# ── 8.2 Save model ────────────────────────────────────────────────────────────
CKPT_PATH = f"{cfg.checkpoint_dir}/nf_tft"
nf.save(path=CKPT_PATH, overwrite=True)
print(f"Model saved → {CKPT_PATH}")


In [ ]:
# ── 8.3 Training curve ────────────────────────────────────────────────────────
def plot_training_curve(nf: NeuralForecast, figsize=(10, 4)):
    """Extract and plot training & validation losses from the NF model."""
    try:
        # NeuralForecast stores trainer logs in the model object
        model   = nf.models[0]
        trainer = model.trainer

        train_losses = [
            x["train_loss"] for x in trainer.callback_metrics.get("train_loss_step", [])
        ] if hasattr(trainer, "callback_metrics") else []

        if not train_losses:
            # Fallback: read from CSV log if logged
            log_dir = Path(CKPT_PATH)
            csv_log = list(log_dir.glob("**/metrics.csv"))
            if csv_log:
                log_df = pd.read_csv(csv_log[0]).dropna(subset=["train_loss"])
                fig, ax = plt.subplots(figsize=figsize)
                ax.plot(log_df["step"], log_df["train_loss"], label="train", lw=1.5)
                if "valid_loss" in log_df:
                    valid = log_df.dropna(subset=["valid_loss"])
                    ax.plot(valid["step"], valid["valid_loss"], label="val", lw=1.5)
                ax.set_xlabel("Step"); ax.set_ylabel("Loss")
                ax.set_title("Training curve")
                ax.legend()
                plt.savefig(f"{cfg.output_dir}/training_curve.png", bbox_inches="tight")
                plt.show()
                return
        print("Training logs not available in this NF version — check Lightning CSV logs.")
    except Exception as e:
        print(f"Could not plot training curve: {e}")


plot_training_curve(nf)


## 9 · Forecasting Strategy & Execution

In [ ]:
# ── 9.1 Generate forecasts ────────────────────────────────────────────────────
forecasts = nf.predict(futr_df=futr_df, static_df=static_df)
print("Forecast shape:", forecasts.shape)
forecasts.head(6)


In [ ]:
# ── 9.2 Merge with actuals ────────────────────────────────────────────────────
pred_col = [c for c in forecasts.columns if c not in ("unique_id", "ds")][0]

eval_df = (
    forecasts
    .merge(test_df[["unique_id", "ds", "y"]], on=["unique_id", "ds"], how="left")
    .merge(feat_df[["unique_id", "ds", "floor_value"]].drop_duplicates(),
           on=["unique_id", "ds"], how="left")
)

eval_df["error"]       = eval_df[pred_col] - eval_df["y"]
eval_df["abs_error"]   = eval_df["error"].abs()
eval_df["below_floor"] = eval_df[pred_col] < eval_df["floor_value"]

mae   = eval_df["abs_error"].mean()
rmse  = (eval_df["error"] ** 2).mean() ** 0.5
viol  = eval_df["below_floor"].mean()

print(f"MAE              : {mae:.4f}")
print(f"RMSE             : {rmse:.4f}")
print(f"Floor viol. rate : {viol:.2%}")


In [ ]:
# ── 9.3 Forecast visualisation ────────────────────────────────────────────────
def plot_forecasts(
    eval_df: pd.DataFrame,
    train_df: pd.DataFrame,
    pred_col: str,
    n_series: int = 4,
    figsize=(16, 10),
):
    """Plot actuals, forecasts, and floor for n_series."""
    uids = eval_df["unique_id"].unique()[:n_series]
    fig, axes = plt.subplots(math.ceil(n_series / 2), 2, figsize=figsize, constrained_layout=True)
    axes = axes.flatten()

    for ax, uid in zip(axes, uids):
        hist = train_df[train_df["unique_id"] == uid].sort_values("ds")
        fore = eval_df[eval_df["unique_id"]  == uid].sort_values("ds")

        ax.plot(hist["ds"], hist["y"], color="steelblue",  label="history",  lw=1.5)
        ax.plot(fore["ds"], fore["y"], color="black",       label="actual",   lw=1.5, ls="--")
        ax.plot(fore["ds"], fore[pred_col], color="darkorange", label="forecast", lw=2)

        if "floor_value" in fore.columns:
            ax.plot(fore["ds"], fore["floor_value"], color="tomato", ls=":",
                    lw=1.5, label="floor")
            ax.fill_between(fore["ds"],
                            fore["floor_value"], fore[pred_col],
                            where=fore[pred_col] < fore["floor_value"],
                            alpha=0.3, color="red", label="violation")

        ax.set_title(uid[:28], fontsize=9)
        ax.legend(fontsize=7)
        ax.tick_params(axis="x", labelsize=7, rotation=20)

    for ax in axes[len(uids):]:
        ax.set_visible(False)

    fig.suptitle(f"TFT Forecasts  |  MAE={mae:.2f}  RMSE={rmse:.2f}  Violations={viol:.1%}",
                 fontsize=11)
    plt.savefig(f"{cfg.output_dir}/forecasts.png", bbox_inches="tight")
    plt.show()


plot_forecasts(eval_df, train_df, pred_col)


In [ ]:
# ── 9.4 Floor violation summary ───────────────────────────────────────────────
def plot_violation_heatmap(eval_df: pd.DataFrame, pred_col: str, figsize=(12, 5)):
    """Heatmap: violation flag per (series × forecast step)."""
    pivot = eval_df.pivot_table(
        index="unique_id",
        columns="ds",
        values="below_floor",
        aggfunc="first",
    ).fillna(0)

    if pivot.shape[0] > 40:
        # Sample for readability
        pivot = pivot.sample(40, random_state=cfg.random_seed)

    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(pivot.astype(float), ax=ax, cmap="Reds", linewidths=0.3,
                cbar_kws={"label": "Below floor"}, yticklabels=True)
    ax.set_title("Floor violation map — forecast horizon (red = prediction below floor)")
    ax.set_xlabel("Date")
    ax.set_ylabel("Series")
    plt.tight_layout()
    plt.savefig(f"{cfg.output_dir}/violation_heatmap.png", bbox_inches="tight")
    plt.show()


plot_violation_heatmap(eval_df, pred_col)

viol_summary = (
    eval_df.groupby("unique_id")
    .agg(n_violations=("below_floor", "sum"),
         violation_rate=("below_floor", "mean"))
    .sort_values("violation_rate", ascending=False)
)
print("Top violating series:")
print(viol_summary.head(10).to_string())


## 10 · Hyperparameter Tuning (Optuna)

In [ ]:
# ── 10.1 Tuning with NeuralForecast + Optuna ──────────────────────────────────
# NeuralForecast supports auto_nbeats / auto_tft style tuning via the Auto* wrappers.
# For TFT we use the AutoTFT class which wraps Optuna internally.

from neuralforecast.auto import AutoTFT
from ray import tune

# ── Search space ─────────────────────────────────────────────────────────────
TFT_SEARCH_SPACE = {
    "hidden_size":    tune.choice([64, 128, 256]),
    "n_head":         tune.choice([2, 4, 8]),
    "dropout":        tune.uniform(0.05, 0.4),
    "learning_rate":  tune.loguniform(1e-4, 1e-2),
    "batch_size":     tune.choice([16, 32, 64]),
    "max_steps":      tune.choice([500, 1000, 2000]),
    # Fixed during tuning
    "h":              cfg.horizon,
    "input_size":     cfg.input_size,
    "hist_exog_list": HIST_EXOG,
    "stat_exog_list": STAT_EXOG,
    "futr_exog_list": FUTR_EXOG,
    "loss":           ExpertFloorLossNixtla(),
    "scaler_type":    cfg.scaler_type,
    "random_seed":    cfg.random_seed,
    "accelerator":    "gpu" if torch.cuda.is_available() else "cpu",
}

print("Search space:")
for k, v in TFT_SEARCH_SPACE.items():
    if not k.endswith("_list") and k != "loss":
        print(f"  {k:20s}: {v}")


In [ ]:
# ── 10.2 Run tuning ───────────────────────────────────────────────────────────
# ⚠️  Set num_samples=1 for a quick smoke-test; use cfg.n_tuning_trials for real tuning.
# Tuning respects GPU availability automatically.

auto_tft = AutoTFT(
    h            = cfg.horizon,
    loss         = ExpertFloorLossNixtla(),
    config       = TFT_SEARCH_SPACE,
    num_samples  = cfg.n_tuning_trials,
    cpus         = 2,
    gpus         = torch.cuda.device_count() if torch.cuda.is_available() else 0,
    refit_with_val = False,
)

nf_tuned = NeuralForecast(models=[auto_tft], freq="MS")

nf_tuned.fit(
    df        = train_df,
    static_df = static_df,
    val_size  = cfg.val_size,
)

print("✅ Tuning complete")


In [ ]:
# ── 10.3 Best config ──────────────────────────────────────────────────────────
try:
    best_cfg = auto_tft.results.get_best_result().config
    print("Best hyperparameters:")
    for k, v in best_cfg.items():
        if not k.endswith("_list") and k != "loss":
            print(f"  {k:20s}: {v}")
except Exception:
    print("Results not available yet — run tuning with num_samples > 0.")


# ── 10.4 Tuning results visualisation ────────────────────────────────────────
def plot_tuning_results(auto_model, figsize=(12, 4)):
    """Plot loss vs. learning_rate and hidden_size from Optuna trials."""
    try:
        df_res = auto_model.results.get_dataframe()
        fig, axes = plt.subplots(1, 2, figsize=figsize)
        axes[0].scatter(df_res["config/learning_rate"], df_res["loss"], alpha=0.6)
        axes[0].set_xscale("log")
        axes[0].set_xlabel("learning_rate"); axes[0].set_ylabel("val loss")
        axes[0].set_title("Loss vs. LR")

        axes[1].scatter(df_res["config/hidden_size"], df_res["loss"], alpha=0.6, c="darkorange")
        axes[1].set_xlabel("hidden_size"); axes[1].set_ylabel("val loss")
        axes[1].set_title("Loss vs. hidden_size")
        plt.tight_layout()
        plt.savefig(f"{cfg.output_dir}/tuning_results.png", bbox_inches="tight")
        plt.show()
    except Exception as e:
        print(f"Could not plot tuning results: {e}")

plot_tuning_results(auto_tft)


## 11 · Explainability
Two lenses: (a) TFT attention weights — *what time-steps matter?*  (b) SHAP — *which features drive predictions?*

In [ ]:
# ── 11.1 TFT Attention weights ────────────────────────────────────────────────
def extract_attention_weights(
    nf_model:  NeuralForecast,
    df:        pd.DataFrame,
    static_df: pd.DataFrame,
    futr_df:   pd.DataFrame,
    n_samples: int = 8,
) -> Optional[pd.DataFrame]:
    """
    NeuralForecast's TFT exposes attention via predict(return_samples=False).
    We use the model's internal interpret() method when available.
    Returns a DataFrame of shape [series, encoder_steps, attention_heads].
    """
    try:
        model = nf_model.models[0]
        if not hasattr(model, "decompose"):
            print("Attention decomposition not available in this NF version.")
            return None

        # Call decompose for a subset of series
        uid_sample = df["unique_id"].unique()[:n_samples]
        sub_df     = df[df["unique_id"].isin(uid_sample)]
        sub_futr   = futr_df[futr_df["unique_id"].isin(uid_sample)]
        sub_stat   = static_df[static_df["unique_id"].isin(uid_sample)]

        result = model.decompose(sub_df, sub_futr, sub_stat)
        return result
    except Exception as e:
        print(f"Attention extraction: {e}")
        return None


attn = extract_attention_weights(nf, train_df, static_df, futr_df)
if attn is not None:
    print("Attention shape:", attn.shape)


In [ ]:
# ── 11.2 Temporal attention heatmap ──────────────────────────────────────────
def plot_attention_heatmap(attn_df, figsize=(14, 5)):
    """Visualise average attention across encoder positions."""
    if attn_df is None:
        print("No attention data — using synthetic example for illustration.")
        # Synthetic attention for illustration
        n_enc  = cfg.input_size
        n_dec  = cfg.horizon
        data   = np.random.dirichlet(np.ones(n_enc), size=(n_dec,))
        fig, ax = plt.subplots(figsize=figsize)
        sns.heatmap(data.T, ax=ax, cmap="YlOrRd", yticklabels=5,
                    xticklabels=[f"h+{i+1}" for i in range(n_dec)],
                    cbar_kws={"label": "Attention weight"})
        ax.set_title("TFT Temporal Attention (encoder ← decoder) — illustrative")
        ax.set_xlabel("Forecast step"); ax.set_ylabel("Encoder position (t-k)")
        plt.tight_layout()
        plt.savefig(f"{cfg.output_dir}/attention_heatmap.png", bbox_inches="tight")
        plt.show()
        return

    avg = attn_df.mean(axis=0)   # average over series & heads
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(avg, ax=ax, cmap="YlOrRd", cbar_kws={"label": "Attention weight"})
    ax.set_title("TFT Temporal Attention (averaged over series)")
    ax.set_xlabel("Forecast step"); ax.set_ylabel("Encoder position (t-k)")
    plt.tight_layout()
    plt.savefig(f"{cfg.output_dir}/attention_heatmap.png", bbox_inches="tight")
    plt.show()


plot_attention_heatmap(attn)


In [ ]:
# ── 11.3 SHAP feature importance on forecasts ─────────────────────────────────
import shap
from sklearn.ensemble import GradientBoostingRegressor

def compute_shap_importance(
    train_df:  pd.DataFrame,
    eval_df:   pd.DataFrame,
    pred_col:  str,
    features:  List[str],
    figsize=(10, 6),
):
    """
    Proxy SHAP: fit a GBM on the same features used by TFT and explain predictions.
    This is a model-agnostic approximation useful for stakeholder communication.
    """
    feature_cols = [f for f in features if f in train_df.columns]
    X_train = train_df[feature_cols + ["y"]].dropna()
    y_train = X_train.pop("y")

    gbm = GradientBoostingRegressor(n_estimators=200, max_depth=4, random_seed=cfg.random_seed
                                    if hasattr(GradientBoostingRegressor, "random_seed")
                                    else 42)
    gbm.set_params(random_state=cfg.random_seed)
    gbm.fit(X_train, y_train)

    # SHAP values
    explainer   = shap.TreeExplainer(gbm)
    X_sample    = X_train.sample(min(500, len(X_train)), random_state=cfg.random_seed)
    shap_values = explainer.shap_values(X_sample)

    # Summary plot
    fig = plt.figure(figsize=figsize)
    shap.summary_plot(shap_values, X_sample, plot_type="bar", show=False)
    plt.title("SHAP Feature Importance (GBM proxy for TFT)")
    plt.tight_layout()
    plt.savefig(f"{cfg.output_dir}/shap_importance.png", bbox_inches="tight")
    plt.show()

    # Return sorted importance
    importance = pd.DataFrame({
        "feature":   feature_cols,
        "shap_mean": np.abs(shap_values).mean(axis=0),
    }).sort_values("shap_mean", ascending=False)
    return importance


shap_df = compute_shap_importance(
    train_df, eval_df, pred_col,
    features=HIST_EXOG + FUTR_EXOG + ["floor_value"],
)
print("Top features by SHAP:")
print(shap_df.head(10).to_string(index=False))


In [ ]:
# ── 11.4 Variable selection weights (TFT native) ─────────────────────────────
def plot_variable_selection(nf_model: NeuralForecast, figsize=(10, 5)):
    """
    TFT learns explicit variable selection weights (VSW).
    If available, plot as a ranked bar chart.
    """
    try:
        model = nf_model.models[0]
        # Access variable selection gates
        vsw = {}
        if hasattr(model, "hist_encoder"):
            for name, param in model.hist_encoder.named_parameters():
                if "var_select" in name:
                    vsw[name] = param.data.mean().item()

        if not vsw:
            print("Variable selection weights not directly accessible via this API version.")
            print("Falling back to SHAP-derived importance (see above).")
            return

        vsw_df = pd.DataFrame(list(vsw.items()), columns=["param", "weight"])                    .sort_values("weight", ascending=True)
        fig, ax = plt.subplots(figsize=figsize)
        ax.barh(vsw_df["param"], vsw_df["weight"], color="steelblue")
        ax.set_title("TFT Variable Selection Weights")
        plt.tight_layout()
        plt.savefig(f"{cfg.output_dir}/variable_selection.png", bbox_inches="tight")
        plt.show()
    except Exception as e:
        print(f"VSW extraction error: {e}")


plot_variable_selection(nf)


## 12 · Summary & Data Requirements

In [ ]:
# ── 12.1 Performance summary table ────────────────────────────────────────────
def performance_summary(eval_df: pd.DataFrame, pred_col: str) -> pd.DataFrame:
    """Return per-contract-type and overall metrics."""
    eval_with_type = eval_df.merge(
        feat_df[["unique_id", "ds", "contract_type"]].drop_duplicates(),
        on=["unique_id", "ds"], how="left"
    )

    def metrics(g):
        err = g[pred_col] - g["y"]
        mae  = g["abs_error"].mean()   if "abs_error" in g.columns else err.abs().mean()
        rmse = (err ** 2).mean() ** 0.5
        viol = (g[pred_col] < g["floor_value"]).mean() if "floor_value" in g.columns else np.nan
        smape = (2 * err.abs() / (g[pred_col].abs() + g["y"].abs() + 1e-9)).mean()
        return pd.Series({"MAE": round(mae,3), "RMSE": round(rmse,3),
                          "sMAPE": round(smape,3), "ViolRate": round(viol,4)})

    per_type = eval_with_type.groupby("contract_type").apply(metrics)
    overall  = metrics(eval_df.assign(**{pred_col: eval_df[pred_col]}))
    overall.name = "ALL"

    summary = pd.concat([per_type, overall.to_frame().T])
    return summary


summary_df = performance_summary(eval_df, pred_col)
print("═" * 55)
print("  MODEL PERFORMANCE SUMMARY")
print("═" * 55)
print(summary_df.to_string())
print("═" * 55)


In [ ]:
# ── 12.2 Metrics radar chart (stakeholder viz) ────────────────────────────────
def plot_radar_summary(summary_df: pd.DataFrame, figsize=(7, 7)):
    """Quick radar chart of normalised metrics per contract type."""
    from matplotlib.patches import FancyArrowPatch

    metrics_cols = ["MAE", "RMSE", "sMAPE", "ViolRate"]
    sub = summary_df[metrics_cols].copy().fillna(0)
    sub_norm = (sub - sub.min()) / (sub.max() - sub.min() + 1e-9)  # 0–1 scale

    labels   = metrics_cols
    n        = len(labels)
    angles   = np.linspace(0, 2 * np.pi, n, endpoint=False).tolist()
    angles  += angles[:1]

    fig, ax = plt.subplots(figsize=figsize, subplot_kw={"polar": True})
    colors  = ["steelblue", "darkorange", "forestgreen", "crimson"]

    for i, (idx, row) in enumerate(sub_norm.iterrows()):
        vals = row.tolist() + row.tolist()[:1]
        ax.plot(angles, vals, "-o", lw=2, label=str(idx), color=colors[i % len(colors)])
        ax.fill(angles, vals, alpha=0.08, color=colors[i % len(colors)])

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels, fontsize=10)
    ax.set_title("Normalised Metrics by Contract Type", pad=15)
    ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.15), fontsize=9)
    plt.tight_layout()
    plt.savefig(f"{cfg.output_dir}/radar_summary.png", bbox_inches="tight")
    plt.show()


plot_radar_summary(summary_df)


## 12.3 Raw Data Requirements for Production (20 000+ Series)

This section documents the **minimum required fields** to run this pipeline at scale.

### Required columns (per row = one month, one series)
| Column | Type | Description | Example |
|--------|------|-------------|---------|
| `date` | `str / int` | Period in `YYYYMM` format | `202401` |
| `ts_id` | `str` | Unique contract / series identifier | `999 R1316P007 513` |
| `actual_cost_paid` | `float` | (**target**) Observed cost for the period | `9.181` |
| `contract_value` | `float` | Reference / benchmark contract value (can be NaN) | `11.21` |

### Strongly recommended enrichment columns
| Column | Type | Description |
|--------|------|-------------|
| `contract_type` | `int` (1/2/3) | Floor type — if unknown, assign via heuristic |
| `v_initial` | `float` | Initial contract value at contract start date |
| `tasa_custom` | `float` | Custom capitalisation rate (only for Type 3) |
| `contract_start` | `date` | First active month of the contract |

### Data quality checklist
- [ ] ≥ 24 months of history per series (model `input_size` = 24)
- [ ] No future data leakage in covariates
- [ ] `contract_value` should be non-null for at least the first period
- [ ] `floor_value` must be computable for all future forecast periods
- [ ] No duplicate `(ts_id, date)` pairs
- [ ] Time series sorted chronologically before loading

### Scale guidance
| # Series | Recommended instance | Expected training time (1 000 steps) |
|-----------|---------------------|---------------------------------------|
| < 5 000 | ml.g4dn.xlarge (1× T4) | ~20 min |
| 5 000 – 20 000 | ml.p3.2xlarge (1× V100) | ~60 min |
| 20 000+ | ml.p3.8xlarge (4× V100) | ~90 min (multi-GPU) |

### Production checklist
- [ ] Data stored in S3 (`s3://bucket/contracts/timeseries_YYYYMM.parquet`)
- [ ] Model artefact saved to S3 after training
- [ ] Inference pipeline triggered monthly via SageMaker Pipelines
- [ ] Floor violations flagged in downstream BI dashboards
- [ ] Retrain trigger: MAE drift > 15 % vs. baseline


In [ ]:
# ── 12.4 Final artefact summary ───────────────────────────────────────────────
import os

artefacts = list(Path(cfg.output_dir).glob("*.png")) + list(Path(cfg.checkpoint_dir).rglob("*"))
print("Generated artefacts:")
for a in sorted(artefacts):
    size = os.path.getsize(a) / 1024
    print(f"  {str(a):<55}  {size:6.1f} KB")

print("\n✅ Notebook execution complete.")
print(f"   Forecast MAE              : {mae:.4f}")
print(f"   Forecast RMSE             : {rmse:.4f}")
print(f"   Floor violation rate      : {viol:.2%}")
print(f"   Series trained            : {train_df['unique_id'].nunique()}")
print(f"   Horizon                   : {cfg.horizon} months")


---
### Appendix A — Production Callback: Gradient-Level Floor Penalty

For full gradient-level integration of `ExpertFloorLoss` into NeuralForecast TFT training,
override the model's `training_step` via a PyTorch Lightning callback:

```python
import pytorch_lightning as pl

class FloorPenaltyCallback(pl.Callback):
    """Adds ExpertFloorLoss penalty on top of the model's default loss."""

    def __init__(self, floor_loss: ExpertFloorLoss, penalty_weight: float = 0.5):
        self.floor_loss     = floor_loss
        self.penalty_weight = penalty_weight

    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx):
        # batch structure depends on NF version; adapt as needed
        # x_features = extract_contract_features(batch)
        # penalty = self.floor_loss(outputs["y_hat"], outputs["y"], x_features)
        # trainer.strategy.reduce(penalty * self.penalty_weight)
        pass

# Usage:
# nf = NeuralForecast(models=[tft_model], freq="MS")
# nf.fit(df=train_df, static_df=static_df, val_size=12,
#        trainer_kwargs={"callbacks": [FloorPenaltyCallback(ExpertFloorLoss())]})
```
